<a href="https://colab.research.google.com/github/internshipinabook/data-science-internshipinabook/blob/main/notebooks/week7_debugging_STARTER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> ⏸️ **Pause and Predict**
> Before running your model: predict whether it will overfit or underfit — and why. What in your data or model choice leads you to that prediction? Check the learning curves after training.

In [ ]:
# ── Colab setup (skip if running locally) ──────────────────────
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/internshipinabook/data-science-internshipinabook.git book
    %cd book
    !pip install -r requirements.txt -q
    print('Colab setup complete ✅')
else:
    print('Running locally ✅')


# Week 7 — Debugging and Cross-Validation
### Week 7 — The Data Science Internship | Kova Analytics

> **STARTER notebook.**

---

## ⚠️ Common Mistakes

| Mistake | What goes wrong | Fix |
|---------|-----------------|-----|
| Treating a high training score as success | A model that scores 0.98 on train and 0.71 on test has memorised the training data, not learned the pattern. | Always compare training and validation scores side by side |
| Fixing overfitting only with regularisation | Regularisation is one tool. Feature selection, cross-validation, and simpler models are often better fixes. | Try reducing features before increasing regularisation strength |

## 🏆 Challenge Task

> 🎯 **Core Path**
> Document three things you tried in this notebook that did not work. For each, write one sentence explaining why. This is your debugging log — it is part of your professional record.

> 🔬 **Advanced Path**
> Plot learning curves for your model. Diagnose the failure mode (overfit/underfit/high variance) and implement one fix. Show before-and-after metrics.

## ✅ What You Can Do After This Week

- Diagnose a performance drop as overfitting, data drift, or data leakage
- Encapsulate the full preparation pipeline in a `prepare_dataset()` function
- Align new data to the training schema using `.reindex(fill_value=0)`
- Run 5-fold stratified cross-validation and interpret mean ± std of recall
- Retrain and evaluate a model on a combined historical + new dataset

*Add `week7_debugging.ipynb` to your internship portfolio.*

<a href="https://colab.research.google.com/github/internshipinabook/data-science-internshipinabook/blob/main/notebooks/week7_debugging_STARTER.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>&nbsp;&nbsp;
<a href="https://github.com/internshipinabook/data-science-internshipinabook/blob/main/notebooks/week7/week7_debugging_STARTER.ipynb">
  <img src="https://img.shields.io/badge/View%20on-GitHub-181717?logo=github" alt="View on GitHub"/>
</a>

---

## 🖥️ How to Run This Notebook

| Platform | Instructions |
|----------|-------------|
| **Google Colab** | Click the badge above — free, no setup needed |
| **Local Jupyter** | `pip install -r requirements.txt` then `jupyter lab` |
| **VS Code** | Open `.ipynb` with the Jupyter extension installed |
| **GitHub** | Click "View on GitHub" above for a read-only preview |

> **STARTER notebook** — Complete the `# YOUR CODE HERE` cells. Check the SOLUTION notebook if stuck.

In [ ]:
# ── PLATFORM SETUP ───────────────────────────────────────────────────────────
# Run this cell first. It installs missing libraries in Google Colab.
# In a local environment, skip it if requirements.txt is already installed.

import sys, subprocess

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("📦 Google Colab detected — installing libraries...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'pandas>=1.5', 'numpy>=1.23', 'matplotlib>=3.6',
                    'seaborn>=0.12', 'scikit-learn>=1.2'], check=True)
    print("✅ Libraries installed.")
else:
    print("💻 Running locally.")
    print("   If you see ImportError below, run: pip install -r requirements.txt")

## 📂 Dataset

The dataset is included in the course repository. No Kaggle account required.

**File:** `customer_churn.csv`  
**Source:** `https://raw.githubusercontent.com/internshipinabook/data-science-internshipinabook/main/data/customer_churn.csv`

Run the cell below to load it directly.

## 🔄 Adaptability — Use Your Own Dataset

This notebook is written for the Telco Churn dataset but the **entire ML pipeline works on any binary classification problem**.

**To swap in your own data, change only these lines at the top of the notebook:**
```python
DATA_FILE      = 'customer_churn.csv'  # ← your CSV filename
TARGET_COL     = 'Churn'              # ← the binary column you want to predict (Yes/No or 1/0)
POSITIVE_LABEL = 'Yes'                # ← which value means "positive" (the event you care about)
ID_COL         = 'customerID'         # ← identifier column to drop before modelling (or None)
NUMERIC_FIX    = 'TotalCharges'       # ← any numeric column stored as text (or None)
```

**Works with any binary classification dataset:**
- 🏦 Loan default prediction
- 🏥 Patient readmission risk
- 📧 Email spam detection
- 💳 Credit card fraud
- 🛒 Customer conversion
- 🎓 Student dropout prediction

> ⚠️ **Class imbalance warning:** If your dataset has <20% positive class, use `class_weight='balanced'` — already done in this notebook. Check your class distribution in Step 4 before proceeding.

---

## 🔄 Using a Different Dataset?

This notebook is written for `customer_churn.csv` but the ML pipeline applies to any binary classification problem.

**To adapt:**
1. Change `DATA_FILE` to your filename
2. Set `TARGET_COL` to your binary target column (the thing you want to predict)
3. Update `POSITIVE_LABEL` to the positive class name (e.g. `'Yes'`, `'1'`, `'Fraud'`)
4. The `prepare_dataset()` function encodes features — update the column references to match yours

```python
# ── Adapt these for your dataset ─────────────────────────────────────────────
DATA_FILE      = 'customer_churn.csv'   # ← change to your file
TARGET_COL     = 'Churn'               # ← your binary target column
POSITIVE_LABEL = 'Yes'                 # ← the positive class value
ID_COL         = 'customerID'          # ← identifier column to drop (or None)
```

**Works with any binary classification dataset** — fraud detection, disease prediction,
loan default, email spam, customer conversion — as long as your target has two classes.

---

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, GridSearchCV, cross_validate)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (recall_score, precision_score, f1_score,
                              classification_report, confusion_matrix,
                              roc_curve, auc, precision_recall_curve)
%matplotlib inline; sns.set_theme(style='whitegrid')
print('✅ Libraries loaded')

In [ ]:
SERVICE_COLS=['PhoneService','MultipleLines','InternetService','OnlineSecurity',
              'OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
def prepare_dataset(df):
    df=df.copy()
    df['TotalCharges']=pd.to_numeric(df['TotalCharges'],errors='coerce')
    df['TotalCharges']=df['TotalCharges'].fillna(0)  # new customers have zero charges
    if 'customerID' in df.columns: df=df.drop(columns=['customerID'])
    le=LabelEncoder(); df['Churn']=le.fit_transform(df['Churn'].astype(str))
    df['tenure_charge_ratio']=df['TotalCharges']/(df['tenure']+1)
    sc=[c for c in SERVICE_COLS if c in df.columns]
    for col in sc: df[f'{col}_binary']=df[col].apply(lambda x: 1 if str(x).lower()=='yes' else 0)
    df['num_services']=df[[f'{c}_binary' for c in sc]].sum(axis=1)
    df['contract_numeric']=df['Contract'].map({'Month-to-month':2,'One year':1,'Two year':0})
    df['paperless_numeric']=(df['PaperlessBilling']=='Yes').astype(int)
    df['contract_paperless_interaction']=df['contract_numeric']*df['paperless_numeric']
    df['tenure_segment']=pd.cut(df['tenure'],bins=[0,12,36,float('inf')],labels=['New','Established','Loyal'])
    return df
print('✅ prepare_dataset() defined')

## Step 1 — Simulate the New Batch (run this cell, then investigate the drift)

In [ ]:
import pandas as pd, os

# Smart loader: works locally (from package) and in Colab (from GitHub)
LOCAL_PATH  = '../data/customer_churn.csv'
REMOTE_URL  = 'https://raw.githubusercontent.com/internshipinabook/data-science-internshipinabook/main/data/customer_churn.csv'
DATA_SOURCE = LOCAL_PATH if os.path.exists(LOCAL_PATH) else REMOTE_URL
df = pd.read_csv(DATA_SOURCE)
print(f'Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns ✅')
print(f'Source: {DATA_SOURCE}')
df.head()


## Step 2 — State Your Hypothesis
**My hypothesis:** 

**My reasoning:** 

## Step 3 — Prepare Both Datasets
*(Apply prepare_dataset to both. Align columns with .reindex(). Reset indices before subgroup analysis.)*

In [ ]:
# YOUR CODE HERE
# CRITICAL: y_test_arr = y_test.reset_index(drop=True).values


## Step 4 — Overfitting Check
Training recall vs test recall vs new batch recall. Is the gap >10pp?

In [ ]:
# YOUR CODE HERE


## Step 5 — Data Drift Check
Compare feature means. Flag >10% shift with ⚠️

In [ ]:
# YOUR CODE HERE


## Step 6 — Temporal Leakage Check
Correlation with tenure. Flag >0.7.

In [ ]:
# YOUR CODE HERE


## Step 7 — 5-Fold Cross-Validation
Mean, std, range. Compare to Week 6 single-split result.

In [ ]:
# YOUR CODE HERE


## Step 8 — CV Stability Chart

In [ ]:
# YOUR CODE HERE


## Step 9 — Multi-Metric CV (recall, precision, F1 — train and test)

In [ ]:
# YOUR CODE HERE


## Step 10 — Retrain on Combined Data

In [ ]:
# YOUR CODE HERE


## Step 11 — Grid Search

> ⏱️ Use `fast_param_grid` first (~1 min). Switch to `full_param_grid` when time allows.

In [ ]:
fast_param_grid={'n_estimators':[100,200],'max_depth':[10,20],'min_samples_leaf':[1,4]}
full_param_grid={'n_estimators':[50,100,200],'max_depth':[None,10,20],
                 'min_samples_split':[2,5,10],'min_samples_leaf':[1,2,4]}
# YOUR CODE HERE — GridSearchCV with fast_param_grid


## Step 12 — Performance Journey Chart (all estimates Wk5→Wk7)

In [ ]:
# YOUR CODE HERE


## Step 13 — Five-Section Debugging Report
### 1 — The Failure
### 2 — The Investigation
### 3 — Root Cause
### 4 — The Fix
### 5 — What We Will Do Differently
*(Must name a specific owner and a specific stage.)*

---
## ✅ Week 7 Complete
**Next:** `week8/*_STARTER.ipynb`

---
*github.com/InternshipInABook*